## 调节效应分析

*清理了所有多余声明与状态错乱问题*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from pathlib import Path
from IPython.display import display as ipy_display

import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
cluster_name_map = {
    0: '广泛试探型',
    1: '高效采纳型',
    2: '审慎批判型',
}

corr_labels_cn = {
        'first_ai_time': '首次调用时间',
        'trial_calls': 'AI调用次数',
        'pre_think_time': '平均调用前思考时间',
        'pre_first_call_ideas': '首次调用前想法数量',
        'originality': '原创性',
        'fluency': '流畅性',
        'above_median': '高质量回答数',
        'above_median_ratio': '高质量回答率',
        'bfi_openness': '开放性',
        'bfi_conscientiousness': '尽责性',
        'bfi_extraversion': '外向性',
        'bfi_agreeableness': '宜人性',
        'bfi_neuroticism': '神经质',
        'ribs_total': 'RIBS总分',
        'cse_total': '创造性自我效能',
        'dat_score': 'DAT分数',
        'ai_attitude': 'AI态度',
        'perspective_taking': '观点采择率',
        'affected_by_ai': '受AI影响率',
        'age': '年龄',
        'gender': '性别'
}

In [ ]:
# 读取并合并数据
output_dir = Path('output')
data_dir = Path('../../../data/analysis')
pickle_dir = Path('../../../data/pickle')

trial_with_cluster_file = output_dir / 'trial_with_cluster.csv'
performance_file = data_dir / 'performance' / 'performance.csv'
questionnaire_file = pickle_dir / 'user_traits.csv'

df_cluster = pd.read_csv(trial_with_cluster_file)
df_perf = pd.read_csv(performance_file)

# 确保预处理正确合并 (原逻辑)
df_merged = pd.merge(df_cluster, df_perf, on=['participant_id', 'trial_index'], how='inner')

if questionnaire_file.exists():
    df_q = pd.read_csv(questionnaire_file)
    df_merged = pd.merge(df_merged, df_q, on='participant_id', how='left')

mod_df = df_merged[df_merged["cluster"] >= 0].copy()

def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

# 对所有参与分析的连续变量做 Z 标准化
numeric_vars_to_z = [
    "trial_calls", "first_ai_time", "pre_think_time", "pre_first_call_ideas",
    "cse_total", "bfi_openness", "ai_attitude", "ribs_total", "dat_score",
    "perspective_taking", "affected_by_ai", "age",
    "originality", "fluency"
]

for col in numeric_vars_to_z:
    if col in mod_df.columns:
        mod_df[f"{col}_z"] = zscore(mod_df[col].astype(float))
        
print(f"最终用于调节效应分析的样本点数量: {len(mod_df)}")

In [ ]:
# 核心计算部分 (单一收卷版本，杜绝重复定义和作用域混淆)

def _fit_mixed_or_ols(formula, data, has_groups, model_label=""):
    """尝试使用 LMM，如果无随机效应分组或者收敛失败则回退至 OLS"""
    if has_groups:
        try:
            model = sm.MixedLM.from_formula(
                formula,
                groups="participant_id",
                vc_formula={"item_name": "0 + C(item_name)"},
                data=data
            )
            result = model.fit(reml=False, method="lbfgs", maxiter=2000)
            return result, "LMM"
        except Exception as e:
            pass # LMM failed, fallback to OLS quietly
    model = ols(formula, data=data).fit()
    return model, "OLS"


def _fit_and_jn(predictor, moderator, outcome, data, controls=["age", "gender"], alpha=0.05, verbose=False):
    """拟合并精准计算 JN 函数的交点与区间（严密的尺度还原）"""
    p_z = f"{predictor}_z" if f"{predictor}_z" in data.columns else predictor
    m_z = f"{moderator}_z" if f"{moderator}_z" in data.columns else moderator

    subset = data.dropna(subset=[p_z, m_z, outcome] + controls).copy()
    if len(subset) < 30:
        return None

    ctrl_str = " + ".join(controls) if controls else ""
    rhs = f"{p_z} * {m_z}" + (f" + {ctrl_str}" if ctrl_str else "")
    formula = f"{outcome} ~ {rhs}"

    has_groups = "participant_id" in subset.columns and "item_name" in subset.columns
    model, mtype = _fit_mixed_or_ols(formula, subset, has_groups)

    # 提取系数与协方差矩阵
    if mtype == "LMM":
        params = model.fe_params
        cov_fe = model.cov_params().loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z} + {ctrl_str}"
        try:
            model_main, _ = _fit_mixed_or_ols(main_formula, subset, has_groups)
            lr_stat = 2 * (model.llf - model_main.llf)
            p_inter = stats.chi2.sf(lr_stat, 1)
        except Exception:
            p_inter = np.nan
    else:
        params = model.params
        cov_fe = model.cov_params()
        if hasattr(cov_fe, "loc"): cov_fe = cov_fe.loc[params.index, params.index]
        main_formula = f"{outcome} ~ {p_z} + {m_z} + {ctrl_str}"
        model_main = ols(main_formula, data=subset).fit()
        anova_res = sm.stats.anova_lm(model_main, model)
        p_inter = anova_res["Pr(>F)"].iloc[1] if "Pr(>F)" in anova_res.columns else np.nan

    # 定位主效应项与交互项
    inter_terms = [k for k in params.index if ":" in k and p_z in k]
    if not inter_terms:
        inter_terms = [k for k in params.index if ":" in k]
    if not inter_terms:
        return None
    inter_key = inter_terms[0]

    b_pred = params.get(p_z)
    b_inter = params.get(inter_key)
    
    if b_pred is None or b_inter is None:
        return None

    try:
        v_pred = cov_fe.at[p_z, p_z]
        v_inter = cov_fe.at[inter_key, inter_key]
        cov_pi = cov_fe.at[p_z, inter_key]
    except KeyError:
        return None

    # JN 精准计算 (彻底解决 Z标度错位问题)
    z_crit = stats.norm.ppf(1 - alpha / 2)

    # m_raw = 用于人类查看画图横坐标的本源刻度, m_model = 输入模型的计算刻度
    m_raw = subset[moderator]
    m_model = subset[m_z]
    
    m_raw_min, m_raw_max = m_raw.min(), m_raw.max()
    m_model_min, m_model_max = m_model.min(), m_model.max()

    # 插值构建连续区间
    m_range = np.linspace(m_raw_min, m_raw_max, 300)
    m_model_range = np.linspace(m_model_min, m_model_max, 300)

    # 简单斜率严格基于模型刻度计算
    simple_slope = b_pred + b_inter * m_model_range
    var_slope = v_pred + m_model_range**2 * v_inter + 2 * m_model_range * cov_pi
    se_slope = np.sqrt(np.maximum(var_slope, 0))
    ci_lo = simple_slope - z_crit * se_slope
    ci_hi = simple_slope + z_crit * se_slope
    sig_mask = (ci_lo > 0) | (ci_hi < 0)

    # 解 JN 二次方程边界点
    A = b_inter**2 - z_crit**2 * v_inter
    B = 2 * (b_pred * b_inter - z_crit**2 * cov_pi)
    C = b_pred**2 - z_crit**2 * v_pred

    jn_points_raw = []
    if abs(A) > 1e-12:
        disc = B**2 - 4 * A * C
        if disc >= 0:
            sqrt_disc = np.sqrt(disc)
            for root_model in [(-B + sqrt_disc) / (2 * A), (-B - sqrt_disc) / (2 * A)]:
                if m_model_min <= root_model <= m_model_max:
                    # 线性等比还原回原值
                    root_raw = np.interp(root_model, [m_model_min, m_model_max], [m_raw_min, m_raw_max])
                    jn_points_raw.append(root_raw)

    return {
        "m_range": m_range, 
        "simple_slope": simple_slope,
        "ci_lo": ci_lo, 
        "ci_hi": ci_hi, 
        "sig_mask": sig_mask,
        "jn_points": sorted(jn_points_raw), 
        "p_inter": p_inter,
        "p_cn": corr_labels_cn.get(predictor, predictor),
        "m_cn": corr_labels_cn.get(moderator, moderator),
        "y_cn": corr_labels_cn.get(outcome, outcome),
        "mtype": mtype, "m_raw": m_raw
    }

def plot_moderation_grid_jn(interactions, data, n_cols=3, controls=["age", "gender"], figsize_per_cell=(4.5, 3.5)):
    """绘制所有 JN 的网格汇总图"""
    valid = []
    for pred, mod, out in interactions:
        res = _fit_and_jn(pred, mod, out, data, controls)
        if res is not None:
            valid.append(res)
            
    n_rows = (len(valid) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize_per_cell[0] * n_cols, figsize_per_cell[1] * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, res in enumerate(valid):
        ax = axes[i]
        
        if res["sig_mask"].any():
            in_sig = False; start = None
            for j, sig in enumerate(res["sig_mask"]):
                if sig and not in_sig:
                    start = res["m_range"][j]
                    in_sig = True
                elif not sig and in_sig:
                    ax.axvspan(start, res["m_range"][j], color="#FFF3B0", alpha=0.6, lw=0)
                    in_sig = False
            if in_sig: ax.axvspan(start, res["m_range"][-1], color="#FFF3B0", alpha=0.6, lw=0)

        ax.fill_between(res["m_range"], res["ci_lo"], res["ci_hi"], color="#2c7bb6", alpha=0.10, lw=0)
        ax.plot(res["m_range"], res["simple_slope"], color="#2c7bb6", linewidth=2)
        ax.axhline(y=0, color="gray", linestyle=":", linewidth=0.8)

        for jp in res["jn_points"]:
            ax.axvline(x=jp, color="#D4A017", linestyle="--", linewidth=1, alpha=0.8)

        ax.set_xlabel(res["m_cn"], fontsize=9)
        ax.set_ylabel(f'{res["p_cn"]} 的简单斜率', fontsize=9)
        ax.set_title(f'{res["p_cn"]} x {res["m_cn"]} → {res["y_cn"]}', fontsize=10, fontweight="bold")
        
        # Add sig box
        # sig_prop = ((res["ci_lo"] > 0) | (res["ci_hi"] < 0)).mean()
        # p_val = res["p_inter"]
        # p_str = f"p = {p_val:.3f}" if p_val >= 0.001 else "p < 0.001"
        # ax.text(0.98, 0.03, f"{p_str} | 显著区间 {sig_prop:.0%}",
        #     transform=ax.transAxes, ha="right", va="bottom",
        #     fontsize=8, bbox=dict(boxstyle="round,pad=0.2", facecolor="wheat", alpha=0.7))

    for j in range(len(valid), len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    return fig

In [ ]:
# ===== 执行绘图分析 =====
sig_trait_interactions = [
    ("first_ai_time",      "cse_total",    "fluency"),
    ("pre_think_time",     "ribs_total",   "fluency"),
    ("pre_think_time",     "ribs_total",   "above_median"),
    ("trial_calls",        "ribs_total",   "originality"),
    ("pre_think_time",     "dat_score",    "originality"),
    ("pre_think_time",     "dat_score",    "above_median_ratio"),
    ("perspective_taking", "dat_score",    "originality"),
    ("perspective_taking", "dat_score",    "above_median_ratio"),
]

print("开始生成汇总 JN 调节效应图 (修正 Z 量纲误差版本)...")
fig = plot_moderation_grid_jn(sig_trait_interactions, mod_df, n_cols=3)
fig.savefig("output/moderation_personality_jn_clean.png", dpi=200, bbox_inches="tight")
plt.show()